In [0]:
describe customers_clean;

col_name,data_type,comment
customer_id,bigint,null
customer_name,string,null
email,string,null
city,string,null
customer_type,string,null


In [0]:
describe orders_clean;

col_name,data_type,comment
order_id,bigint,null
customer_id,double,null
order_date,timestamp,null
status,string,null


In [0]:
describe order_items_clean;

col_name,data_type,comment
order_item_id,bigint,null
order_id,bigint,null
product_id,bigint,null
quantity,bigint,null
discount_percent,double,null


In [0]:
describe products_clean;

col_name,data_type,comment
product_id,bigint,null
product_name,string,null
category,string,null
unit_price,double,null


## 1. total revenue per category

calculate the total revenue generated for each product category.

**formula:**

revenue = quantity × unit_price × (1 - discount_percent / 100)

In [0]:
select
    p.category,
    round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as total_revenue
from order_items_clean oi
join products_clean p
    on oi.product_id = p.product_id
group by p.category
order by total_revenue desc;

category,total_revenue
Home,2.552194715E7
Clothing,2.42025982E7
Electronics,2.354265855E7
Books,2.284417017E7


## 2. top 10 customers by total order value

find the top 10 customers based on the total value of their orders.

In [0]:
select
    c.customer_id,
    c.customer_name,
    round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as total_order_value
from customers_clean c
join orders_clean o
    on c.customer_id = o.customer_id
join order_items_clean oi
    on o.order_id = oi.order_id
join products_clean p
    on oi.product_id = p.product_id
group by c.customer_id, c.customer_name
order by total_order_value desc
limit 10;

customer_id,customer_name,total_order_value
90,Mary Rogers,1008669.37
153,Michael Stephens,877703.76
488,Robert Hines,778658.22
764,Amber Johnson,696819.78
6,Monica Herrera,673382.76
760,Stephen Garcia,633665.99
321,Manuel King,620344.96
86,Matthew Smith,611674.53
362,Charles Price,605667.32
331,Jennifer Ayala,589297.37


## 3. month-wise order count for the last 12 months

calculate the number of orders placed in each month for the last 12 months.

In [0]:
select
    date_format(order_date, 'yyyy-MM') as month,
    count(*) as total_orders
from orders_clean
where order_date >= add_months(current_date(), -12)
group by date_format(order_date, 'yyyy-MM')
order by month;

month,total_orders
2025-07,14
2025-08,41
2025-09,39
2025-10,35
2025-11,43
2025-12,43
2026-01,45
2026-02,29
2026-03,33
2026-04,21


## 4. find customers who placed orders but never had any item delivered

find customers who have placed one or more orders, but none of their orders were delivered.

In [0]:
select distinct
    c.customer_id,
    c.customer_name
from customers_clean c
join orders_clean o
    on c.customer_id = o.customer_id
where c.customer_id not in (
    select customer_id
    from orders_clean
    where status = 'DELIVERED'
) limit 10;

customer_id,customer_name
12,Jennifer Rocha
23,Lisa Brandt
53,Matthew Ross
64,Angel Riggs
99,Beth Oneill
129,Michael Santos
153,Michael Stephens
160,Jodi Walker
184,Eric Morgan
199,Ralph Lee


## 5. products that were ordered but had more returns than purchases

find the products where the total returned quantity is greater than the total purchased quantity.

In [0]:
select
    p.product_id,
    p.product_name,
    sum(case when oi.quantity > 0 then oi.quantity else 0 end) as purchased_quantity,
    abs(sum(case when oi.quantity < 0 then oi.quantity else 0 end)) as returned_quantity
from order_items_clean oi
join products_clean p
    on oi.product_id = p.product_id
group by p.product_id, p.product_name
having abs(sum(case when oi.quantity < 0 then oi.quantity else 0 end))
       > sum(case when oi.quantity > 0 then oi.quantity else 0 end);

product_id,product_name,purchased_quantity,returned_quantity
173,Formal Shirt,1,4
264,Dell Laptop,2,4
333,Machine Learning Basics,1,3
404,Cap,0,4
344,Curtains,0,4
70,Water Bottle,2,4


## 6. calculate the return rate per category

calculate the return rate for each product category.

**formula:**

return rate = returned items / total items

In [0]:
select
    p.category,
    round(abs(sum(case when oi.quantity < 0 then oi.quantity else 0 end))/ sum(abs(oi.quantity)),2) as return_rate
from order_items_clean oi
join products_clean p
    on oi.product_id = p.product_id
group by p.category
order by return_rate desc;

category,return_rate
Home,0.03
Books,0.03
Electronics,0.03
Clothing,0.03


## 7. running totals with window functions

calculate the running total of revenue over time based on the order date.

In [0]:
with daily_revenue as (
    select
        o.order_date,
        round(
            sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)),
            2
        ) as revenue
    from orders_clean o
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    group by o.order_date
)

select
    order_date,
    revenue,
    sum(revenue) over(order by order_date) as running_total
from daily_revenue
order by order_date limit 10;

order_date,revenue,running_total
2024-07-18T19:16:10.000Z,141761.39,141761.39
2024-07-19T03:57:55.000Z,268347.68,410109.07
2024-07-20T20:59:27.000Z,257145.23,667254.3
2024-07-21T07:57:04.000Z,65273.13,732527.43
2024-07-23T00:05:50.000Z,36691.29,769218.7200000001
2024-07-23T10:42:22.000Z,317310.82,1086529.54
2024-07-24T08:39:31.000Z,43655.91,1130185.45
2024-07-25T16:08:57.000Z,236604.33,1366789.78
2024-07-25T21:30:24.000Z,62250.42,1429040.2
2024-07-27T01:46:54.000Z,232043.87,1661084.0699999998


## 8. ranking with dense_rank

rank customers based on their total order value using dense_rank.

In [0]:
with customer_revenue as (
    select
        c.customer_id,
        c.customer_name,
        round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as total_revenue
    from customers_clean c
    join orders_clean o
        on c.customer_id = o.customer_id
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    group by c.customer_id, c.customer_name

)

select
    customer_id,
    customer_name,
    total_revenue,
    dense_rank() over(order by total_revenue desc) as customer_rank
from customer_revenue limit 10;

customer_id,customer_name,total_revenue,customer_rank
90,Mary Rogers,1008669.37,1
153,Michael Stephens,877703.76,2
488,Robert Hines,778658.22,3
764,Amber Johnson,696819.78,4
6,Monica Herrera,673382.76,5
760,Stephen Garcia,633665.99,6
321,Manuel King,620344.96,7
86,Matthew Smith,611674.53,8
362,Charles Price,605667.32,9
331,Jennifer Ayala,589297.37,10


## 9. lag/lead analysis

compare each customer's order value with their previous order using the lag function.

In [0]:
with order_revenue as (

    select
        o.customer_id,
        o.order_id,
        o.order_date,
        round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as order_value
    from orders_clean o
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    where o.customer_id <> -1
    group by
        o.customer_id,
        o.order_id,
        o.order_date

)

select
    customer_id,
    order_id,
    order_date,
    order_value,
    lag(order_value) over(
        partition by customer_id
        order by order_date
    ) as previous_order_value
from order_revenue limit 10;

customer_id,order_id,order_date,order_value,previous_order_value
2.0,329,2025-05-25T03:11:07.000Z,119123.81,null
2.0,778,2025-07-26T17:55:56.000Z,19583.7,119123.81
3.0,340,2025-04-05T23:27:17.000Z,121779.99,null
4.0,402,2026-03-16T03:30:11.000Z,120349.5,null
5.0,164,2025-07-10T07:07:33.000Z,63189.95,null
6.0,170,2024-07-28T15:54:08.000Z,205691.21,null
6.0,715,2024-11-30T09:35:41.000Z,216082.69,205691.21
6.0,366,2025-09-11T08:42:13.000Z,87804.08,216082.69
6.0,656,2026-07-09T13:52:03.000Z,163804.78,87804.08
7.0,567,2025-02-14T14:47:16.000Z,15483.07,null


## 10. cte with multiple levels

find the total revenue generated by each customer and classify them based on their spending.

- high spender : revenue >= 10000
- medium spender : revenue between 5000 and 9999
- low spender : revenue < 5000

In [0]:
with customer_revenue as (

    select
        c.customer_id,
        c.customer_name,
        round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as total_revenue
    from customers_clean c
    join orders_clean o
        on c.customer_id = o.customer_id
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    where c.customer_id <> -1
    group by
        c.customer_id,
        c.customer_name

),

customer_category as (

    select
        customer_id,
        customer_name,
        total_revenue,
        case
            when total_revenue >= 10000 then 'high spender'
            when total_revenue >= 5000 then 'medium spender'
            else 'low spender'
        end as customer_type
    from customer_revenue

)

select *
from customer_category
order by total_revenue desc limit 10;

customer_id,customer_name,total_revenue,customer_type
90,Mary Rogers,1008669.37,high spender
153,Michael Stephens,877703.76,high spender
488,Robert Hines,778658.22,high spender
764,Amber Johnson,696819.78,high spender
6,Monica Herrera,673382.76,high spender
760,Stephen Garcia,633665.99,high spender
321,Manuel King,620344.96,high spender
86,Matthew Smith,611674.53,high spender
362,Charles Price,605667.32,high spender
331,Jennifer Ayala,589297.37,high spender


## 11. ntile for segmentation

divide customers into 4 segments based on their total revenue using the ntile function.

In [0]:
with customer_revenue as (

    select
        c.customer_id,
        c.customer_name,
        round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as total_revenue
    from customers_clean c
    join orders_clean o
        on c.customer_id = o.customer_id
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    where c.customer_id <> -1
    group by
        c.customer_id,
        c.customer_name

)

select
    customer_id,
    customer_name,
    total_revenue,
    ntile(35) over(order by total_revenue desc) as customer_segment
from customer_revenue limit 20;

customer_id,customer_name,total_revenue,customer_segment
90,Mary Rogers,1008669.37,1
153,Michael Stephens,877703.76,1
488,Robert Hines,778658.22,1
764,Amber Johnson,696819.78,1
6,Monica Herrera,673382.76,1
760,Stephen Garcia,633665.99,1
321,Manuel King,620344.96,1
86,Matthew Smith,611674.53,1
362,Charles Price,605667.32,1
331,Jennifer Ayala,589297.37,1


## 12. year-over-year comparison

compare the total revenue of each year with the previous year.

In [0]:
with yearly_revenue as (

    select
        year(o.order_date) as year,
        round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as total_revenue
    from orders_clean o
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    group by year(o.order_date)

)

select
    year,
    total_revenue,
    lag(total_revenue) over(order by year) as previous_year_revenue
from yearly_revenue;

year,total_revenue,previous_year_revenue
2024,2.172837048E7,null
2025,4.844850684E7,2.172837048E7
2026,2.593449675E7,4.844850684E7


## 13. first/last value analysis

find the first and last order value for each customer.

In [0]:
with order_revenue as (
    select
        o.customer_id,
        o.order_id,
        o.order_date,
        round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as order_value
    from orders_clean o
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    where o.customer_id <> -1
    group by
        o.customer_id,
        o.order_id,
        o.order_date

)

select
    customer_id,
    order_id,
    order_date,
    order_value,

    first_value(order_value) over(
        partition by customer_id
        order by order_date
    ) as first_order_value,

    last_value(order_value) over(
        partition by customer_id
        order by order_date
        rows between unbounded preceding and unbounded following
    ) as last_order_value

from order_revenue limit 10;

customer_id,order_id,order_date,order_value,first_order_value,last_order_value
2.0,329,2025-05-25T03:11:07.000Z,119123.81,119123.81,19583.7
2.0,778,2025-07-26T17:55:56.000Z,19583.7,119123.81,19583.7
3.0,340,2025-04-05T23:27:17.000Z,121779.99,121779.99,121779.99
4.0,402,2026-03-16T03:30:11.000Z,120349.5,120349.5,120349.5
5.0,164,2025-07-10T07:07:33.000Z,63189.95,63189.95,63189.95
6.0,170,2024-07-28T15:54:08.000Z,205691.21,205691.21,163804.78
6.0,715,2024-11-30T09:35:41.000Z,216082.69,205691.21,163804.78
6.0,366,2025-09-11T08:42:13.000Z,87804.08,205691.21,163804.78
6.0,656,2026-07-09T13:52:03.000Z,163804.78,205691.21,163804.78
7.0,567,2025-02-14T14:47:16.000Z,15483.07,15483.07,94788.2


## 14. cumulative distribution

calculate the cumulative distribution of customers based on their total revenue using the cume_dist function.

In [0]:
with customer_revenue as (

    select
        c.customer_id,
        c.customer_name,
        round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)), 2) as total_revenue
    from customers_clean c
    join orders_clean o
        on c.customer_id = o.customer_id
    join order_items_clean oi
        on o.order_id = oi.order_id
    join products_clean p
        on oi.product_id = p.product_id
    where c.customer_id <> -1
    group by
        c.customer_id,
        c.customer_name

)

select
    customer_id,
    customer_name,
    total_revenue,
    cume_dist() over(order by total_revenue desc) as cumulative_distribution
from customer_revenue limit 10;

customer_id,customer_name,total_revenue,cumulative_distribution
90,Mary Rogers,1008669.37,0.0018867924528301887
153,Michael Stephens,877703.76,0.0037735849056603774
488,Robert Hines,778658.22,0.005660377358490566
764,Amber Johnson,696819.78,0.007547169811320755
6,Monica Herrera,673382.76,0.009433962264150943
760,Stephen Garcia,633665.99,0.011320754716981131
321,Manuel King,620344.96,0.013207547169811321
86,Matthew Smith,611674.53,0.01509433962264151
362,Charles Price,605667.32,0.016981132075471698
331,Jennifer Ayala,589297.37,0.018867924528301886


## 15. complex cte: cohort analysis

group customers based on the month of their first order and count how many customers belong to each cohort.

In [0]:
with first_order as (

    select
        customer_id,
        min(order_date) as first_order_date
    from orders_clean
    where customer_id <> -1
    group by customer_id

),

customer_cohort as (

    select
        customer_id,
        date_format(first_order_date, 'yyyy-MM') as cohort_month
    from first_order

)

select
    cohort_month,
    count(customer_id) as total_customers
from customer_cohort
group by cohort_month
order by cohort_month;

cohort_month,total_customers
2024-07,20
2024-08,36
2024-09,29
2024-10,30
2024-11,38
2024-12,34
2025-01,23
2025-02,27
2025-03,18
2025-04,30


## 16. self-join with window function

find each customer's next order date by joining the orders table with itself.

In [0]:
with ordered_orders as (

    select
        customer_id,
        order_id,
        order_date,
        row_number() over(
            partition by customer_id
            order by order_date
        ) as order_no
    from orders_clean
    where customer_id <> -1

)

select
    o1.customer_id,
    o1.order_id as current_order,
    o1.order_date as current_order_date,
    o2.order_id as next_order,
    o2.order_date as next_order_date
from ordered_orders o1
left join ordered_orders o2
    on o1.customer_id = o2.customer_id
    and o1.order_no + 1 = o2.order_no
order by
    o1.customer_id,
    o1.order_no limit 10;

customer_id,current_order,current_order_date,next_order,next_order_date
2.0,329,2025-05-25T03:11:07.000Z,778,2025-07-26T17:55:56.000Z
2.0,778,2025-07-26T17:55:56.000Z,null,null
3.0,340,2025-04-05T23:27:17.000Z,null,null
4.0,402,2026-03-16T03:30:11.000Z,null,null
5.0,164,2025-07-10T07:07:33.000Z,null,null
6.0,170,2024-07-28T15:54:08.000Z,715,2024-11-30T09:35:41.000Z
6.0,715,2024-11-30T09:35:41.000Z,366,2025-09-11T08:42:13.000Z
6.0,366,2025-09-11T08:42:13.000Z,656,2026-07-09T13:52:03.000Z
6.0,656,2026-07-09T13:52:03.000Z,null,null
7.0,567,2025-02-14T14:47:16.000Z,405,2026-03-13T02:12:26.000Z
